# Final Experiments for DAMK-Net / MK-MNet

Notebook nay de chay cac thuc nghiem bo sung tren Google Colab voi structure repo moi.

Muc tieu:
- multi-seed augmentation comparison
- validation-first threshold selection
- Grad-CAM cho operating point chinh
- optional lightweight profiling

Notebook co sanity checks de tranh loi path, checkpoint, dataset, va output folder.

## Cach dung nhanh

1. Upload repo `lightweight-medical-model` len Google Drive, hoac clone vao Drive.
2. Dat dataset BUSI vao dung path trong cell config.
3. Dat checkpoint `.pt` vao dung path trong cell config.
4. Chay notebook tu tren xuong duoi.

Neu muon chi chay mot phan, bat/tat cac co `RUN_*` trong cell config.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Toggle tung nhom experiment
RUN_AUGMENTATION = True
RUN_THRESHOLD_PROTOCOL = True
RUN_GRADCAM = True
RUN_PROFILING = False

# Thu cac candidate root de tim repo cho do met vi path Drive
PROJECT_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/MS_HCMUS/AppliedML'),
    Path('/content/drive/MyDrive/AppliedML'),
    Path('/content/drive/MyDrive'),
]

REPO_NAME = 'lightweight-medical-model'

# Neu dataset va checkpoint khong nam trong repo thi sua 2 path nay cho khop Drive cua ban
DATASET_DIR_CANDIDATES = [
    'data/busi',
    '../data/busi',
    '/content/drive/MyDrive/MS_HCMUS/AppliedML/data/busi',
    '/content/drive/MyDrive/AppliedML/data/busi',
]

CHECKPOINT_CANDIDATES = [
    'outputs/busi/mk_mnet/img_224/width_0.5/oversample_1/lambda_0.8/lr_0.0003/weight_decay_0.0001/patience_10/best_model.pt',
    '/content/drive/MyDrive/MS_HCMUS/AppliedML/checkpoints/mk_mnet_width05_os_best_model.pt',
    '/content/drive/MyDrive/AppliedML/checkpoints/mk_mnet_width05_os_best_model.pt',
]

AUG_POLICIES = ['none', 'rotate', 'translate_y', 'scale', 'horizontal_flip']
AUG_SEEDS = [42, 123, 2026]

MODEL_NAME = 'mk_mnet'
WIDTH_MULT = 0.5
PROFILE_MODELS = ['mednet', 'mk_mnet', 'r_cbam_mnet']
PROFILE_WIDTH_MULTS = [0.25, 0.5, 1.0]
PROFILE_ITERATIONS = 100

In [ ]:
import os
import shutil
import subprocess
import sys
from pprint import pprint


def find_repo_root() -> Path:
    direct_candidates = []
    for root in PROJECT_ROOT_CANDIDATES:
        direct_candidates.append(root / REPO_NAME)
        direct_candidates.append(root)

    for candidate in direct_candidates:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists():
            return candidate

    for root in PROJECT_ROOT_CANDIDATES:
        if not root.exists():
            continue
        matches = list(root.rglob('pyproject.toml'))
        for match in matches:
            parent = match.parent
            if parent.name == REPO_NAME and (parent / 'scripts').exists():
                return parent

    raise FileNotFoundError(
        'Khong tim thay repo lightweight-medical-model. Hay sua PROJECT_ROOT_CANDIDATES cho dung Drive cua ban.'
    )


def resolve_path(repo_root: Path, candidates) -> Path:
    for item in candidates:
        path = Path(item)
        if not path.is_absolute():
            path = repo_root / path
        if path.exists():
            return path
    raise FileNotFoundError(f'Khong tim thay path hop le trong candidates: {candidates}')


def run(cmd, cwd=None):
    print('\n>>>', ' '.join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=cwd, check=True)


def ensure_uv():
    if shutil.which('uv'):
        print('uv da san sang')
        return
    run([sys.executable, '-m', 'pip', 'install', '-U', 'uv'])


def print_tree(root: Path, max_depth=2):
    root = root.resolve()
    print(f'\nTree: {root}')
    for path in sorted(root.rglob('*')):
        depth = len(path.relative_to(root).parts)
        if depth > max_depth:
            continue
        prefix = '  ' * (depth - 1)
        suffix = '/' if path.is_dir() else ''
        print(f'{prefix}{path.name}{suffix}')

In [ ]:
REPO_ROOT = find_repo_root()
DATASET_DIR = resolve_path(REPO_ROOT, DATASET_DIR_CANDIDATES)
CHECKPOINT_PATH = resolve_path(REPO_ROOT, CHECKPOINT_CANDIDATES)

AUG_OUTPUT_ROOT = REPO_ROOT / 'artifacts' / 'augmentation-comparison'
THRESHOLD_OUTPUT_ROOT = REPO_ROOT / 'artifacts' / 'threshold-validation-protocol' / 'mk_mnet_width05_os'
GRADCAM_OUTPUT_ROOT = REPO_ROOT / 'artifacts' / 'gradcam'
PROFILE_OUTPUT_CSV = REPO_ROOT / 'artifacts' / 'model-profiles' / 'profile.csv'

print('REPO_ROOT      =', REPO_ROOT)
print('DATASET_DIR    =', DATASET_DIR)
print('CHECKPOINT     =', CHECKPOINT_PATH)
print('AUG_OUTPUT     =', AUG_OUTPUT_ROOT)
print('THRESH_OUTPUT  =', THRESHOLD_OUTPUT_ROOT)
print('GRADCAM_OUTPUT =', GRADCAM_OUTPUT_ROOT)
print('PROFILE_CSV    =', PROFILE_OUTPUT_CSV)

In [ ]:
assert (REPO_ROOT / 'pyproject.toml').exists(), 'Thieu pyproject.toml trong repo root'
assert (REPO_ROOT / 'scripts').exists(), 'Thieu folder scripts'
assert DATASET_DIR.exists(), f'Dataset dir khong ton tai: {DATASET_DIR}'
assert CHECKPOINT_PATH.exists(), f'Checkpoint khong ton tai: {CHECKPOINT_PATH}'

print('Sanity checks OK')
print_tree(REPO_ROOT / 'scripts', max_depth=2)

In [ ]:
ensure_uv()
run(['uv', 'sync'], cwd=REPO_ROOT)

## 1. Multi-seed augmentation comparison

In [ ]:
if RUN_AUGMENTATION:
    cmd = [
        'uv', 'run', 'python', '-m', 'scripts.experiments.run_augmentation_comparison',
        '--dataset-dir', str(DATASET_DIR),
        '--policies', *AUG_POLICIES,
        '--seeds', *[str(seed) for seed in AUG_SEEDS],
        '--resume',
    ]
    run(cmd, cwd=REPO_ROOT)
else:
    print('Bo qua augmentation comparison')

In [ ]:
import pandas as pd

summary_csv = AUG_OUTPUT_ROOT / 'comparison_summary.csv'
status_csv = AUG_OUTPUT_ROOT / 'comparison_status.csv'

if summary_csv.exists():
    display(pd.read_csv(summary_csv))
else:
    print('Chua co comparison_summary.csv')

if status_csv.exists():
    display(pd.read_csv(status_csv))
else:
    print('Chua co comparison_status.csv')

## 2. Validation-first threshold selection

In [ ]:
if RUN_THRESHOLD_PROTOCOL:
    cmd = [
        'uv', 'run', 'python', '-m', 'scripts.experiments.run_threshold_validation_protocol',
        '--model', MODEL_NAME,
        '--width-mult', str(WIDTH_MULT),
        '--checkpoint', str(CHECKPOINT_PATH),
        '--dataset-dir', str(DATASET_DIR),
        '--output-root', str(THRESHOLD_OUTPUT_ROOT),
        '--resume',
    ]
    run(cmd, cwd=REPO_ROOT)
else:
    print('Bo qua threshold validation protocol')

In [ ]:
selected_summary = THRESHOLD_OUTPUT_ROOT / 'selected_threshold_summary.csv'
val_sweep = THRESHOLD_OUTPUT_ROOT / 'val' / 'threshold_sensitivity.csv'
test_eval = THRESHOLD_OUTPUT_ROOT / 'test' / 'threshold_sensitivity.csv'

for path in [selected_summary, val_sweep, test_eval]:
    print('\n', path)
    if path.exists():
        display(pd.read_csv(path))
    else:
        print('Chua co file')

## 3. Grad-CAM for the main operating point

In [ ]:
if RUN_GRADCAM:
    cmd = [
        'uv', 'run', 'python', '-m', 'scripts.visualization.generate_gradcam',
        '--model', MODEL_NAME,
        '--width-mult', str(WIDTH_MULT),
        '--checkpoint', str(CHECKPOINT_PATH),
        '--dataset-dir', str(DATASET_DIR),
        '--outputs-root', str(GRADCAM_OUTPUT_ROOT),
        '--all-samples',
        '--target-class', 'true',
    ]
    run(cmd, cwd=REPO_ROOT)
else:
    print('Bo qua Grad-CAM')

In [ ]:
from IPython.display import Image, display

gradcam_candidates = list(GRADCAM_OUTPUT_ROOT.rglob('*.png'))
print(f'Tim thay {len(gradcam_candidates)} file PNG trong {GRADCAM_OUTPUT_ROOT}')

for image_path in gradcam_candidates[:8]:
    print(image_path)
    display(Image(filename=str(image_path), width=480))

## 4. Optional profiling

In [ ]:
if RUN_PROFILING:
    cmd = [
        'uv', 'run', 'python', '-m', 'scripts.analysis.profile_models',
        '--models', *PROFILE_MODELS,
        '--width-mults', *[str(w) for w in PROFILE_WIDTH_MULTS],
        '--iterations', str(PROFILE_ITERATIONS),
        '--output', str(PROFILE_OUTPUT_CSV),
    ]
    run(cmd, cwd=REPO_ROOT)
else:
    print('Bo qua profiling')

In [ ]:
if PROFILE_OUTPUT_CSV.exists():
    display(pd.read_csv(PROFILE_OUTPUT_CSV))
else:
    print('Chua co profile.csv')

## Output de dua vao report

- `artifacts/augmentation-comparison/comparison_summary.csv`: bang mean/std cho augmentation.
- `artifacts/threshold-validation-protocol/.../selected_threshold_summary.csv`: chon tau tren validation va metric test sau khi khoa tau.
- `artifacts/gradcam/.../*.png`: hinh dinh tinh cho operating point chinh.
- `artifacts/model-profiles/profile.csv`: params, FLOPs, latency local.

Sau khi chay xong, download cac CSV va PNG can thiet tu Drive de cap nhat lai report.